# Investra Data Preprocessing

Notebook ini untuk memproses data mentah menjadi CSV final yang kompatibel dengan backend INVESTRA.

## 1) Setup dan Path

In [ ]:
from pathlib import Path
from datetime import datetime
import re

import numpy as np
import pandas as pd

print('pandas:', pd.__version__)

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    WORK_DIR = cwd
elif (cwd.parent / 'data').exists():
    WORK_DIR = cwd.parent
else:
    raise RuntimeError('Folder data tidak ditemukan. Jalankan notebook dari folder investra-gov-apps-data-prep atau subfolder notebooks.')

RAW_DIR = WORK_DIR / 'data' / 'raw'
INTERIM_DIR = WORK_DIR / 'data' / 'interim'
FINAL_DIR = WORK_DIR / 'data' / 'final'

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print('WORK_DIR :', WORK_DIR)
print('RAW_DIR  :', RAW_DIR)
print('INTERIM  :', INTERIM_DIR)
print('FINAL    :', FINAL_DIR)

## 2) Baca Data Mentah (`.csv` atau `.xlsx`)

Jika `INPUT_FILE = None`, notebook otomatis ambil file pertama dari `data/raw`.

In [ ]:
INPUT_FILE = None  # contoh: 'raw_dataset_2024.xlsx'

if INPUT_FILE is None:
    candidates = sorted(
        list(RAW_DIR.glob('*.csv')) +
        list(RAW_DIR.glob('*.xlsx')) +
        list(RAW_DIR.glob('*.xls'))
    )
    if not candidates:
        raise FileNotFoundError('Tidak ada file mentah di data/raw. Taruh file .csv/.xlsx dulu.')
    input_path = candidates[0]
else:
    input_path = RAW_DIR / INPUT_FILE
    if not input_path.exists():
        raise FileNotFoundError(f'File tidak ditemukan: {input_path}')

if input_path.suffix.lower() == '.csv':
    df_raw = pd.read_csv(input_path)
elif input_path.suffix.lower() in {'.xlsx', '.xls'}:
    df_raw = pd.read_excel(input_path)
else:
    raise ValueError('Format file tidak didukung. Gunakan .csv/.xlsx/.xls')

print('Input file:', input_path.name)
print('Shape raw :', df_raw.shape)
df_raw.head()

## 3) Normalisasi Nama Kolom dan Mapping Alias

In [ ]:
REQUIRED_COLUMNS = [
    'provinsi',
    'pmdn_rp',
    'fdi_rp',
    'pdrb_per_kapita',
    'ipm',
    'kemiskinan',
    'akses_listrik',
    'tpt',
]

def normalize_col_name(name: str) -> str:
    s = str(name).strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

ALIASES = {
    'province': 'provinsi',
    'nama_provinsi': 'provinsi',
    'prov': 'provinsi',
    'pmdn': 'pmdn_rp',
    'pmdn_triliun': 'pmdn_rp',
    'fdi': 'fdi_rp',
    'pma_rp': 'fdi_rp',
    'pdrb_perkapita': 'pdrb_per_kapita',
    'pdrb': 'pdrb_per_kapita',
    'akses_listrik_persen': 'akses_listrik',
    'pengangguran': 'tpt',
    'pengangguran_terbuka': 'tpt',
}

df = df_raw.copy()
df.columns = [normalize_col_name(c) for c in df.columns]
df = df.rename(columns={c: ALIASES.get(c, c) for c in df.columns})

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f'Kolom wajib belum lengkap: {missing}. Kolom yang ada: {list(df.columns)}')

df = df[REQUIRED_COLUMNS].copy()
print('Kolom final:', list(df.columns))
df.head()

## 4) Cleaning dan Validasi

In [ ]:
# Clean nama provinsi
df['provinsi'] = (
    df['provinsi']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Drop baris provinsi kosong
df = df[df['provinsi'] != ''].copy()

# Konversi numerik
numeric_cols = [c for c in REQUIRED_COLUMNS if c != 'provinsi']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Report missing
missing_counts = df[numeric_cols].isna().sum()
print('Missing values per kolom:')
display(missing_counts.to_frame('missing_count'))

# Drop rows yang masih null di numeric
before_drop = len(df)
df = df.dropna(subset=numeric_cols).copy()
print(f'Drop NA rows: {before_drop - len(df)}')

# Jika ada duplikat provinsi, pakai rata-rata per provinsi
dup_count = df['provinsi'].duplicated().sum()
if dup_count > 0:
    print(f'Ditemukan duplikat provinsi: {dup_count}. Akan di-aggregate mean per provinsi.')
    df = (
        df.groupby('provinsi', as_index=False)[numeric_cols]
        .mean()
    )

# Rule validasi rentang
for col in ['ipm', 'kemiskinan', 'akses_listrik', 'tpt']:
    df[col] = df[col].clip(lower=0, upper=100)

for col in ['pmdn_rp', 'fdi_rp', 'pdrb_per_kapita']:
    df[col] = df[col].clip(lower=0)

# Sort
df = df.sort_values('provinsi').reset_index(drop=True)

print('Shape after clean:', df.shape)
df.head(10)

## 5) Simpan Interim dan Final CSV

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
interim_path = INTERIM_DIR / f'investra_interim_{ts}.csv'
final_path = FINAL_DIR / f'investra_final_{ts}.csv'

df.to_csv(interim_path, index=False)
df.to_csv(final_path, index=False)

print('Interim saved:', interim_path)
print('Final saved  :', final_path)

## 6) Ringkasan Data Final

In [ ]:
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'non_null': [int(df[c].notna().sum()) for c in df.columns],
})

display(summary)
display(df.describe(include='all').transpose())